# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding chosen #1 — "What Predicts Health?" (ML Appendix, Random Forest feature importance)**

The paper reports Average Position (43%), Impressions (32%), Scroll Depth (15%), and CTR (8%) as
the top predictors of Health Score, and is careful to note this itself: "the target is partly
constructed from some of these inputs, so importance is descriptive rather than causal."

**My methodology question:** Health Score is *defined* as impressions(30pts) + position(30pts) +
CTR(20pts) + scroll(20pts) -- and those same four quantities are exactly the top four features
the Random Forest ranks as most important, together accounting for 98% of the reported importance
(see the code cell below). If a model is trained to predict a score from the literal ingredients
of that score's formula, high importance on those ingredients isn't really a finding about what
*drives* content health -- it's closer to confirming the model can read its own answer key. The
paper already flags this as "descriptive not causal," which is honest, but I'd gently push
further: does this appendix page add information beyond just restating the Health Score formula
in bar-chart form? A cleaner version might predict a genuinely external outcome (future traffic
change, editorial time saved) from these features instead.

**Finding chosen #2 — "What Predicts Growth?" (ML Appendix, Logistic Regression)**

The paper reports 71% holdout accuracy for a logistic regression separating growing from
declining pages.

**My methodology question:** Finding #1 elsewhere in the same paper reports 74.8K growing pages
vs. 45.6K declining pages -- a majority class of about 62.1%. A model that always guessed "growing"
would score ~62% accuracy without learning anything. The reported 71% is a real improvement over
that (about +9 points), but the paper doesn't state this baseline anywhere near the 71% figure, so
a reader can't tell at a glance whether 71% represents strong signal or a fairly modest lift over
just guessing the majority class. I'd ask: could a majority-class (or simple frequency) baseline be
reported side-by-side with the model number, the same way our own Week-4/Week-5 work compares a
rule baseline to a model? That single addition would make the 71% figure much easier to trust at
face value.


In [1]:
# Quantifying both critiques with real numbers, not just impressions of the paper.

# --- Finding #1: how much of the RF importance is literally the Health Score formula? ---
rf_importance = {"Average Position": 43, "Impressions": 32, "Scroll Depth": 15, "CTR": 8, "Clicks": 2}
formula_components = rf_importance["Average Position"] + rf_importance["Impressions"] + \
    rf_importance["Scroll Depth"] + rf_importance["CTR"]
print("Health Score formula: impressions(30) + position(30) + CTR(20) + scroll(20)")
print(f"Share of RF importance coming from literal formula components: {formula_components}%")

# --- Finding #2: majority-class baseline vs the paper's reported 71% accuracy ---
up, down = 74_800, 45_600  # from the paper's own Finding #1 table
majority_baseline_acc = up / (up + down)
paper_reported_acc = 0.71
print(f"\nmajority-class ('always predict growing') baseline accuracy: {majority_baseline_acc:.3f}")
print(f"paper's reported logistic regression accuracy: {paper_reported_acc}")
print(f"actual lift over the unstated baseline: {paper_reported_acc - majority_baseline_acc:+.3f}")


Health Score formula: impressions(30) + position(30) + CTR(20) + scroll(20)
Share of RF importance coming from literal formula components: 98%

majority-class ('always predict growing') baseline accuracy: 0.621
paper's reported logistic regression accuracy: 0.71
actual lift over the unstated baseline: +0.089


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Before:** a plain random 75/25 row-level split -- the default most people reach for first, and
what I'd have done without the client-holdout lesson from Week 5.

**After:** `GroupShuffleSplit` by `client_id` (identical setup to Week 5), so no client's pages
appear in both train and test.


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
eligible["is_declining_label"] = (eligible["trend_direction"] == "down").astype(int)

features = ["impressions_90d", "clicks_90d", "sessions_90d", "avg_position", "ctr",
            "content_age_days", "days_since_last_update", "word_count",
            "engagement_rate", "scroll_rate"]

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))[:k]
    return np.asarray(labels)[order].mean()

# --- BEFORE: plain random split, ignores client grouping ---
X, y = eligible[features].fillna(0), eligible["is_declining_label"]
X_tr, X_te, y_tr, y_te, idx_tr, idx_te = train_test_split(
    X, y, eligible.index, test_size=0.25, random_state=42, stratify=y
)
model_before = GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42)
model_before.fit(X_tr, y_tr)
probs_before = model_before.predict_proba(X_te)[:, 1]
p50_before = precision_at_k(probs_before, y_te.values, 50)
auc_before = roc_auc_score(y_te, probs_before)
overlap_before = set(eligible.loc[idx_tr, "client_id"]) & set(eligible.loc[idx_te, "client_id"])

print("BEFORE -- random row-level split")
print(f"  client overlap between train/test: {len(overlap_before)} of {eligible['client_id'].nunique()} total clients")
print(f"  Precision@50: {p50_before:.3f}   AUC: {auc_before:.3f}")

# --- AFTER: grouped split by client_id (same as w05) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(eligible, groups=eligible["client_id"]))
train, test = eligible.iloc[train_idx], eligible.iloc[test_idx]
overlap_after = set(train["client_id"]) & set(test["client_id"])

X_train, y_train = train[features].fillna(0), train["is_declining_label"]
X_test, y_test = test[features].fillna(0), test["is_declining_label"]
model_after = GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42)
model_after.fit(X_train, y_train)
probs_after = model_after.predict_proba(X_test)[:, 1]
p50_after = precision_at_k(probs_after, y_test.values, 50)
auc_after = roc_auc_score(y_test, probs_after)

print("\nAFTER -- client-grouped split")
print(f"  client overlap between train/test: {len(overlap_after)} (must be 0)")
print(f"  Precision@50: {p50_after:.3f}   AUC: {auc_after:.3f}")

print(f"\nBefore/after gap: Precision@50 drops {p50_before - p50_after:+.3f}, "
      f"AUC drops {auc_before - auc_after:+.3f}, once client leakage is removed.")


BEFORE -- random row-level split
  client overlap between train/test: 31 of 32 total clients
  Precision@50: 0.940   AUC: 0.756



AFTER -- client-grouped split
  client overlap between train/test: 0 (must be 0)
  Precision@50: 0.760   AUC: 0.614

Before/after gap: Precision@50 drops +0.180, AUC drops +0.142, once client leakage is removed.


**What this means:** the random split lets the model partly memorize client-specific quirks
(31 of 32 clients appear in both train and test), which inflates both metrics substantially --
Precision@50 goes from a suspiciously high 0.94 down to a more honest 0.76 once client overlap is
removed. This is close to a full replay of the exact lesson from the paper critique above: an
easy validation design can make a model look far better than it is. The 0.76 grouped-split number
is the one I'm keeping and reporting going forward -- it matches what I already reported in w05.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*


In [3]:
# Same leakage checklist from the w03 lane guide, applied to the 10 features used in w05/w06.

corrs = eligible[features + ["is_declining_label"]].corr()["is_declining_label"] \
    .drop("is_declining_label").sort_values(key=abs, ascending=False)
print("feature correlation with the label (no single feature should be near +-1.0):")
print(corrs)

# is any feature literally one of the fields trend_direction is built from?
label_source_fields = ["impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
                        "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d", "trend_pct"]
overlap = [f for f in features if f in label_source_fields]
print(f"\nfeatures that are literally part of the label's own construction: {overlap} (must be empty)")

# partial-overlap check: impressions_90d is a 90-day aggregate that CONTAINS the 30d/prev-30d
# windows trend_direction is computed from -- not a hard leak, but worth checking how correlated
# it is with the literal trend_pct value the label is thresholded from.
print(f"\ncorrelation of impressions_90d with trend_pct (the number the label is thresholded on): "
      f"{eligible['impressions_90d'].corr(eligible['trend_pct']):.3f}")
print("(weak correlation -- impressions_90d is not acting as a disguised copy of trend_pct)")

# confirm no FlyRank product flags or w04 baseline/reason_code/action columns were used as features
product_flags = ["health_score", "priority_score", "action_type", "needs_ctr_fix", "is_quick_win",
                  "reason_code", "action", "baseline_action_score"]
print(f"\nproduct flags / baseline outputs used as model features: "
      f"{[f for f in features if f in product_flags]} (must be empty)")


feature correlation with the label (no single feature should be near +-1.0):
content_age_days         -0.163882
word_count                0.090157
days_since_last_update    0.081383
ctr                      -0.061911
clicks_90d               -0.039680
avg_position             -0.029035
sessions_90d             -0.023141
impressions_90d          -0.018175
engagement_rate          -0.012743
scroll_rate              -0.002958
Name: is_declining_label, dtype: float64

features that are literally part of the label's own construction: [] (must be empty)

correlation of impressions_90d with trend_pct (the number the label is thresholded on): 0.024
(weak correlation -- impressions_90d is not acting as a disguised copy of trend_pct)

product flags / baseline outputs used as model features: [] (must be empty)


**Audit result:** no hard leaks found. No feature is literally one of the fields
`trend_direction` is computed from, and no FlyRank product flag or my own w04 baseline output
was fed in as a feature. The one thing worth naming honestly: `impressions_90d` is a 90-day
aggregate that technically contains the same 30-day windows the label is built from, so it isn't
*fully* independent of the label in a strict sense -- but its correlation with the literal
`trend_pct` value (0.024) is weak enough that I don't think it's functioning as a disguised
version of the label. Worth re-checking if a future version of this model's performance looks
suspiciously strong again.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (from w05, Section 4):** "Gradient Boosting beats the w04 baseline on Precision@50
(0.76 vs 0.62), so it earns its place over the simpler rule."

That sentence states a comparison as settled fact, without naming the split it depends on or
acknowledging that the underlying label is itself a proxy, not a verified outcome.

**Rewritten, safe language:** "On this client-held-out split, and using a same-window proxy
label, Gradient Boosting's ranked top-50 was directionally more precise than the w04 rule
(0.76 vs 0.62 Precision@50) -- an observed result on this dataset and split, not a guarantee the
gap holds on new clients, future months, or a stricter future-window label. This is
decision-support evidence for preferring the model over the rule here, not proof the model has
learned something causal about why pages decline."


## Receipts

*Save a small JSON summary of this audit's findings.*

In [4]:
import json, os

summary = {
    "paper_findings_critiqued": [
        "ML Appendix - Random Forest predicting Health Score (98% of importance is the score's own formula components)",
        "ML Appendix - Logistic Regression growth prediction (71% accuracy vs an unstated ~62% majority-class baseline)",
    ],
    "split_before_after": {
        "random_split": {"note": "client overlap present, inflated metrics"},
        "grouped_split": {"note": "zero client overlap, matches w05 reported numbers"},
    },
    "leakage_audit": "no hard leaks found in the 10-feature set; impressions_90d has weak (0.024) correlation with trend_pct",
    "claim_rewrite": "w05's baseline-vs-model claim rewritten with split + proxy-label caveats",
}

os.makedirs("work/outputs", exist_ok=True)
with open("work/outputs/w06_validation_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("wrote work/outputs/w06_validation_summary.json")
summary


wrote work/outputs/w06_validation_summary.json


{'paper_findings_critiqued': ["ML Appendix - Random Forest predicting Health Score (98% of importance is the score's own formula components)",
  'ML Appendix - Logistic Regression growth prediction (71% accuracy vs an unstated ~62% majority-class baseline)'],
 'split_before_after': {'random_split': {'note': 'client overlap present, inflated metrics'},
  'grouped_split': {'note': 'zero client overlap, matches w05 reported numbers'}},
 'leakage_audit': 'no hard leaks found in the 10-feature set; impressions_90d has weak (0.024) correlation with trend_pct',
 'claim_rewrite': "w05's baseline-vs-model claim rewritten with split + proxy-label caveats"}

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.